# 小组交互入口

请保留整个仓库文件夹；完整教学入口为 notebooks/00_project_demo.ipynb。

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'bs6221_flow').is_dir())
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
torch.set_num_threads(4)
from bs6221_flow.sampling import load_flow, generate_images, compare_solvers, solve_ode
from bs6221_flow.training import train_model
from bs6221_flow.presentation import show_training, forward_demo, probability_paths, generation_demo, diffusion_demo, show_saved_numerics
from bs6221_flow.widgets import generation_panel, numerics_panel
from IPython.display import display
DEVICE = os.environ.get('BS6221_DEVICE', 'auto')
FLOW_WEIGHTS = None  # None: frozen 15,000-step EMA; or your training run's best.pt
model = load_flow(FLOW_WEIGHTS, DEVICE)
print('Device:', next(model.parameters()).device, '| weights SHA-256:', model.flow_weight_sha256)

## 2. 选择数字和噪声，再看生成过程
选择数字、噪声种子、求解器和步数。先预览噪声，再生成图片；相同模型、标签和初始噪声可重复。
相同 NFE 比较会使用同一噪声；改变噪声尺度属于扰动实验，默认标准高斯尺度为 1。
实际路径来自积分过程，不是图片插值。可调用 generate_images(..., noise=你的张量, net=model)。

In [ ]:
playground = generation_panel(model)
display(playground)

## 3. ODE 数值分析：参数、成本和收敛性
NFE = 步数 × 每步网络调用数：Euler 1、Heun 2、RK4 4。相同步数与相同 NFE 是不同预算。
下面先展示原冻结模型的实验记录；它们不是这次重新计算的结果，也不对应自训练模型。
理论阶需要正则性和渐近步长条件。RMSE 相对于近似参考解，不是真实图片误差。
稳定性测试中的线性刚性不证明这个神经网络 ODE 刚性。

In [ ]:
show_saved_numerics()

### 自定义小规模数值实验（点击才计算）
改变样本数、步数列表、NFE、参考分辨率与计时重复。各方法使用同一个当前模型、标签和噪声。
CPU float64 RK4 两级加密检验参考分辨率；未通过时不要报告可信收敛阶。
小样本适合探索，正式结论应使用完整独立样本与更精细参考。

In [ ]:
ode_panel = numerics_panel(model)
display(ode_panel)